# auto_arima 메모리 & 속도 비교 — rustima vs pmdarima

`데이터적용_예측.ipynb`의 시간 비교에 더해 **프로세스 RSS 피크**, **Python 힙(tracemalloc) 피크**, **wall time**을 함께 측정.

- **RSS** : Rust(rustima 내부) + Python(pmdarima) 양쪽 할당을 모두 포함 — 50ms 간격으로 별도 스레드가 폴링하여 피크 기록
- **tracemalloc** : Python 힙만 (rustima의 Rust 측 할당은 잡히지 않음 — 참고용)
- **baseline 보정** : 각 측정 시작 시점의 RSS를 기준으로 `Δ = peak − baseline` 산출 (RSS는 한 번 커지면 줄지 않으므로 절대값 비교는 무의미)

In [1]:
import os, time, threading, gc, tracemalloc, warnings
import numpy as np
import polars as pl
import psutil

warnings.filterwarnings('ignore')


class Profiler:
    """RSS peak (process-wide) + tracemalloc peak (python heap) + wall time."""

    def __init__(self, interval=0.05):
        self.interval = interval
        self.proc = psutil.Process(os.getpid())

    def __enter__(self):
        gc.collect()
        self.rss_baseline = self.proc.memory_info().rss
        self.rss_peak = self.rss_baseline
        self._stop = False
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()
        tracemalloc.start()
        self.t0 = time.perf_counter()
        return self

    def _sample(self):
        while not self._stop:
            rss = self.proc.memory_info().rss
            if rss > self.rss_peak:
                self.rss_peak = rss
            time.sleep(self.interval)

    def __exit__(self, *exc):
        self.elapsed = time.perf_counter() - self.t0
        _, self.tracemalloc_peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        self._stop = True
        self._thread.join()
        self.rss_delta = self.rss_peak - self.rss_baseline

    def report(self, label):
        mb = 1024 ** 2
        print(f"{label:<10} time={self.elapsed:7.3f}s  "
              f"RSS Δpeak={self.rss_delta/mb:7.1f} MB  "
              f"tracemalloc peak={self.tracemalloc_peak/mb:7.1f} MB")

In [3]:
# 데이터 로드 (파주 지점, 최근 60일 = 1,440h)
df = pl.read_csv("/Users/icy71/GitHub/Rust-python-arima/rustima/0_Example_cy/total_heat_demand_dwh.csv")
df_paju = (
    df.filter(pl.col('branch_id') == '파주')
      .with_columns(pl.col('date').str.to_datetime())
      .sort('date')
)
y_sub = df_paju['heat_demand'].tail(60 * 24).to_numpy().astype(np.float64)
print(f"비교 데이터: n={len(y_sub)}h, s=24")

비교 데이터: n=1440h, s=24


## [A] rustima auto_arima — 메모리/시간 측정

In [5]:
from rustima import auto_arima as rs_auto

with Profiler() as p_rs:
    auto_rs = rs_auto(y_sub, s=24, trend='n', stepwise=True, trace=False)

rs_order    = auto_rs.order
rs_seasonal = auto_rs.seasonal_order
rs_aic      = float(auto_rs.result.aic)

p_rs.report("rustima")
print(f"           order={rs_order}{rs_seasonal}  AIC={rs_aic:.2f}")

rustima    time= 29.788s  RSS Δpeak=  126.8 MB  tracemalloc peak=   60.6 MB
           order=(1, 0, 1)(1, 1, 1, 24)  AIC=8853.09


## [B] pmdarima auto_arima — 메모리/시간 측정

In [6]:
import pmdarima as pm

with Profiler() as p_pm:
    auto_pm = pm.auto_arima(
        y_sub,
        seasonal=True, m=24, trend=None,
        stepwise=True, suppress_warnings=True,
    )

pm_order    = auto_pm.order
pm_seasonal = auto_pm.seasonal_order
pm_aic      = float(auto_pm.aic())

p_pm.report("pmdarima")
print(f"           order={pm_order}{pm_seasonal}  AIC={pm_aic:.2f}")

pmdarima   time=179.002s  RSS Δpeak= 6078.6 MB  tracemalloc peak=16531.8 MB
           order=(4, 1, 1)(2, 0, 0, 24)  AIC=9190.66


## [C] 최종 비교 — 속도 / 메모리 / AIC

In [7]:
def fmt_mb(b):
    return f"{b/1024**2:7.1f} MB"

print("=" * 86)
print(f"{'engine':<10}{'order':<14}{'seasonal':<16}"
      f"{'AIC':>10}{'time(s)':>10}{'RSS Δpeak':>14}{'PyHeap peak':>14}")
print("-" * 86)
print(f"{'rustima':<10}{str(rs_order):<14}{str(rs_seasonal):<16}"
      f"{rs_aic:>10.2f}{p_rs.elapsed:>10.3f}{fmt_mb(p_rs.rss_delta):>14}{fmt_mb(p_rs.tracemalloc_peak):>14}")
print(f"{'pmdarima':<10}{str(pm_order):<14}{str(pm_seasonal):<16}"
      f"{pm_aic:>10.2f}{p_pm.elapsed:>10.3f}{fmt_mb(p_pm.rss_delta):>14}{fmt_mb(p_pm.tracemalloc_peak):>14}")
print("=" * 86)

speedup   = p_pm.elapsed / p_rs.elapsed if p_rs.elapsed > 0 else float('inf')
rss_ratio = p_pm.rss_delta / p_rs.rss_delta if p_rs.rss_delta > 0 else float('inf')
heap_ratio = p_pm.tracemalloc_peak / p_rs.tracemalloc_peak if p_rs.tracemalloc_peak > 0 else float('inf')

print(f"속도        : rustima 가 {speedup:6.1f}x 빠름  (절감 {p_pm.elapsed - p_rs.elapsed:+.2f}s)")
print(f"RSS 피크 비 : pmdarima / rustima = {rss_ratio:6.2f}x")
print(f"PyHeap 비   : pmdarima / rustima = {heap_ratio:6.2f}x   (rustima의 Rust 측 할당은 미포함)")
print(f"AIC 차      : pmdarima - rustima = {pm_aic - rs_aic:+.2f}")
print(f"order       : {'동일' if (rs_order == pm_order and rs_seasonal == pm_seasonal) else '다름'}")

engine    order         seasonal               AIC   time(s)     RSS Δpeak   PyHeap peak
--------------------------------------------------------------------------------------
rustima   (1, 0, 1)     (1, 1, 1, 24)      8853.09    29.788      126.8 MB       60.6 MB
pmdarima  (4, 1, 1)     (2, 0, 0, 24)      9190.66   179.002     6078.6 MB    16531.8 MB
속도        : rustima 가    6.0x 빠름  (절감 +149.21s)
RSS 피크 비 : pmdarima / rustima =  47.95x
PyHeap 비   : pmdarima / rustima = 272.87x   (rustima의 Rust 측 할당은 미포함)
AIC 차      : pmdarima - rustima = +337.57
order       : 다름


---
## 더 큰 데이터

In [10]:
# 데이터 로드 (파주 지점, 최근 1년 = 8,760h)
y_sub2 = df_paju['heat_demand'].tail(365 * 24).to_numpy().astype(np.float64)
print(f"비교 데이터: n={len(y_sub2)}h, s=24")

비교 데이터: n=8760h, s=24


In [11]:
from rustima import auto_arima as rs_auto

with Profiler() as p_rs:
    auto_rs = rs_auto(y_sub2, s=24, trend='n', stepwise=True, trace=False)

rs_order    = auto_rs.order
rs_seasonal = auto_rs.seasonal_order
rs_aic      = float(auto_rs.result.aic)

p_rs.report("rustima")
print(f"           order={rs_order}{rs_seasonal}  AIC={rs_aic:.2f}")

rustima    time=532.405s  RSS Δpeak=   94.8 MB  tracemalloc peak=    9.1 MB
           order=(4, 0, 5)(1, 1, 2, 24)  AIC=52094.58


In [12]:
import pmdarima as pm

with Profiler() as p_pm:
    auto_pm = pm.auto_arima(
        y_sub2,
        seasonal=True, m=24, trend=None,
        stepwise=True, suppress_warnings=True,
    )

pm_order    = auto_pm.order
pm_seasonal = auto_pm.seasonal_order
pm_aic      = float(auto_pm.aic())

p_pm.report("pmdarima")
print(f"           order={pm_order}{pm_seasonal}  AIC={pm_aic:.2f}")

KeyboardInterrupt: 

In [ ]:
def fmt_mb(b):
    return f"{b/1024**2:7.1f} MB"

print("=" * 86)
print(f"{'engine':<10}{'order':<14}{'seasonal':<16}"
      f"{'AIC':>10}{'time(s)':>10}{'RSS Δpeak':>14}{'PyHeap peak':>14}")
print("-" * 86)
print(f"{'rustima':<10}{str(rs_order):<14}{str(rs_seasonal):<16}"
      f"{rs_aic:>10.2f}{p_rs.elapsed:>10.3f}{fmt_mb(p_rs.rss_delta):>14}{fmt_mb(p_rs.tracemalloc_peak):>14}")
print(f"{'pmdarima':<10}{str(pm_order):<14}{str(pm_seasonal):<16}"
      f"{pm_aic:>10.2f}{p_pm.elapsed:>10.3f}{fmt_mb(p_pm.rss_delta):>14}{fmt_mb(p_pm.tracemalloc_peak):>14}")
print("=" * 86)

speedup   = p_pm.elapsed / p_rs.elapsed if p_rs.elapsed > 0 else float('inf')
rss_ratio = p_pm.rss_delta / p_rs.rss_delta if p_rs.rss_delta > 0 else float('inf')
heap_ratio = p_pm.tracemalloc_peak / p_rs.tracemalloc_peak if p_rs.tracemalloc_peak > 0 else float('inf')

print(f"속도        : rustima 가 {speedup:6.1f}x 빠름  (절감 {p_pm.elapsed - p_rs.elapsed:+.2f}s)")
print(f"RSS 피크 비 : pmdarima / rustima = {rss_ratio:6.2f}x")
print(f"PyHeap 비   : pmdarima / rustima = {heap_ratio:6.2f}x   (rustima의 Rust 측 할당은 미포함)")
print(f"AIC 차      : pmdarima - rustima = {pm_aic - rs_aic:+.2f}")
print(f"order       : {'동일' if (rs_order == pm_order and rs_seasonal == pm_seasonal) else '다름'}")

---
## 1년 Grid search

In [4]:
# 데이터 로드 (파주 지점, 최근 1년 = 8,760h)
y_sub2 = df_paju['heat_demand'].tail(365 * 24).to_numpy().astype(np.float64)
print(f"비교 데이터: n={len(y_sub2)}h, s=24")

비교 데이터: n=8760h, s=24


In [5]:
from rustima import auto_arima as rs_auto

with Profiler() as p_rs:
    auto_rs = rs_auto(y_sub2, s=24, trend='n', stepwise=False, trace=False)

rs_order    = auto_rs.order
rs_seasonal = auto_rs.seasonal_order
rs_aic      = float(auto_rs.result.aic)

p_rs.report("rustima")
print(f"           order={rs_order}{rs_seasonal}  AIC={rs_aic:.2f}")

rustima    time=856.806s  RSS Δpeak=  260.0 MB  tracemalloc peak=   68.2 MB
           order=(5, 0, 4)(2, 1, 2, 24)  AIC=52067.20


In [ ]:
import pmdarima as pm

with Profiler() as p_pm:
    auto_pm = pm.auto_arima(
        y_sub2,
        seasonal=True, m=24, trend=None,
        stepwise=False, suppress_warnings=True,
    )

pm_order    = auto_pm.order
pm_seasonal = auto_pm.seasonal_order
pm_aic      = float(auto_pm.aic())

p_pm.report("pmdarima")
print(f"           order={pm_order}{pm_seasonal}  AIC={pm_aic:.2f}")

In [ ]:
def fmt_mb(b):
    return f"{b/1024**2:7.1f} MB"

print("=" * 86)
print(f"{'engine':<10}{'order':<14}{'seasonal':<16}"
      f"{'AIC':>10}{'time(s)':>10}{'RSS Δpeak':>14}{'PyHeap peak':>14}")
print("-" * 86)
print(f"{'rustima':<10}{str(rs_order):<14}{str(rs_seasonal):<16}"
      f"{rs_aic:>10.2f}{p_rs.elapsed:>10.3f}{fmt_mb(p_rs.rss_delta):>14}{fmt_mb(p_rs.tracemalloc_peak):>14}")
print(f"{'pmdarima':<10}{str(pm_order):<14}{str(pm_seasonal):<16}"
      f"{pm_aic:>10.2f}{p_pm.elapsed:>10.3f}{fmt_mb(p_pm.rss_delta):>14}{fmt_mb(p_pm.tracemalloc_peak):>14}")
print("=" * 86)

speedup   = p_pm.elapsed / p_rs.elapsed if p_rs.elapsed > 0 else float('inf')
rss_ratio = p_pm.rss_delta / p_rs.rss_delta if p_rs.rss_delta > 0 else float('inf')
heap_ratio = p_pm.tracemalloc_peak / p_rs.tracemalloc_peak if p_rs.tracemalloc_peak > 0 else float('inf')

print(f"속도        : rustima 가 {speedup:6.1f}x 빠름  (절감 {p_pm.elapsed - p_rs.elapsed:+.2f}s)")
print(f"RSS 피크 비 : pmdarima / rustima = {rss_ratio:6.2f}x")
print(f"PyHeap 비   : pmdarima / rustima = {heap_ratio:6.2f}x   (rustima의 Rust 측 할당은 미포함)")
print(f"AIC 차      : pmdarima - rustima = {pm_aic - rs_aic:+.2f}")
print(f"order       : {'동일' if (rs_order == pm_order and rs_seasonal == pm_seasonal) else '다름'}")

---
## [D] rustima 단독 반복 측정 (pmdarima 제외)

pmdarima가 서버를 터뜨려서 제외하고, rustima 만 **N회 반복** 실행해 시간/메모리의 분포를 본다.

- 첫 호출은 JIT/Rust 정적 초기화로 더 느릴 수 있어 **warmup 1회** 후 본 측정 N회
- 각 회차의 wall time, RSS Δpeak, PyHeap peak를 모은 뒤 min/median/mean/max 출력
- 모든 회차에서 동일한 `order/seasonal_order/AIC`가 나와야 정상 (탐색 결정성 확인)

In [ ]:
# 반복 측정 파라미터
N_WARMUP = 1
N_RUNS   = 5

from rustima import auto_arima as rs_auto

# warmup
for _ in range(N_WARMUP):
    _ = rs_auto(y_sub, s=24, trend='n', stepwise=True, trace=False)

# 본 측정
runs = []   # list of dicts
for i in range(N_RUNS):
    with Profiler() as p:
        res = rs_auto(y_sub, s=24, trend='n', stepwise=True, trace=False)
    runs.append({
        "run":      i + 1,
        "time":     p.elapsed,
        "rss":      p.rss_delta,
        "pyheap":   p.tracemalloc_peak,
        "order":    res.order,
        "seasonal": res.seasonal_order,
        "aic":      float(res.result.aic),
    })
    print(f"  run {i+1}/{N_RUNS}  "
          f"time={p.elapsed:7.3f}s  "
          f"RSS Δ={p.rss_delta/1024**2:6.1f} MB  "
          f"PyHeap={p.tracemalloc_peak/1024**2:6.1f} MB  "
          f"AIC={float(res.result.aic):.2f}")

In [ ]:
# 통계 요약
import statistics as stats

def summarize(name, vals, fmt):
    print(f"  {name:<14} min={fmt(min(vals))}  median={fmt(stats.median(vals))}  "
          f"mean={fmt(stats.mean(vals))}  max={fmt(max(vals))}")

times   = [r["time"]   for r in runs]
rss     = [r["rss"]    for r in runs]
pyheap  = [r["pyheap"] for r in runs]

f_s  = lambda v: f"{v:7.3f}s"
f_mb = lambda v: f"{v/1024**2:7.1f}MB"

print("=" * 78)
print(f"rustima auto_arima — {N_RUNS}회 반복 (warmup {N_WARMUP}회 제외)")
print("-" * 78)
summarize("wall time",   times,  f_s)
summarize("RSS Δpeak",   rss,    f_mb)
summarize("PyHeap peak", pyheap, f_mb)
print("-" * 78)

# 결정성 체크
orders     = {r["order"]    for r in runs}
seasonals  = {r["seasonal"] for r in runs}
aics       = {round(r["aic"], 4) for r in runs}
print(f"  order      : {orders}        {'✓ 일관' if len(orders) == 1 else '✗ 불일치'}")
print(f"  seasonal   : {seasonals}        {'✓ 일관' if len(seasonals) == 1 else '✗ 불일치'}")
print(f"  AIC (반올림 1e-4): {aics}        {'✓ 일관' if len(aics) == 1 else '✗ 불일치'}")
print("=" * 78)